# Module 02-3: 3노드 텍스트 분류기

## 학습 목표

1. **키워드 기반 분류**: 딕셔너리의 키워드를 활용한 텍스트 카테고리 분류 로직을 구현한다
2. **3노드 파이프라인**: 전처리 → 분류 → 결과 포맷의 선형 파이프라인을 구성한다
3. **CATEGORY_KEYWORDS 설계**: 카테고리별 키워드 매핑 딕셔너리를 정의한다
4. **텍스트 전처리**: 소문자 변환, 공백 정리 등 기본적인 텍스트 정규화를 수행한다
5. **분류 신뢰도 계산**: 키워드 매칭 수를 기반으로 신뢰도 점수를 산출한다

---
> 참고 문서: `docs/part1-foundations/02-langgraph-fundamentals.md`

## 개념 요약

### 텍스트 분류기 흐름

```
START → [preprocess] → [classify] → [format_result] → END
```

| 노드 | 역할 | 입력 | 출력 |
|------|------|------|------|
| preprocess | 텍스트 정규화 | raw_text | processed_text |
| classify | 카테고리 분류 | processed_text | category, confidence |
| format_result | 결과 포맷 | category, confidence | result |

### 키워드 기반 분류 패턴

```python
CATEGORY_KEYWORDS = {
    "기술": ["AI", "코드", "개발", "소프트웨어"],
    "경제": ["주식", "금리", "경제", "투자"],
    "스포츠": ["축구", "야구", "올림픽", "선수"]
}
```

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("환경 설정 완료")

## 실습 1: ClassifierState 및 CATEGORY_KEYWORDS 정의

텍스트 분류기의 상태와 키워드 딕셔너리를 정의하세요.

**ClassifierState 필드:**
- `raw_text`: 원본 입력 텍스트
- `processed_text`: 전처리된 텍스트
- `category`: 분류된 카테고리
- `confidence`: 신뢰도 (0.0 ~ 1.0)
- `result`: 최종 결과 문자열

**CATEGORY_KEYWORDS:**
- `"기술"`: `["AI", "인공지능", "코드", "개발", "소프트웨어", "알고리즘", "프로그램"]`
- `"경제"`: `["주식", "금리", "경제", "투자", "시장", "환율", "GDP"]`
- `"스포츠"`: `["축구", "야구", "농구", "올림픽", "선수", "경기", "우승"]`
- `"기타"`: `[]`

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: ClassifierState는 TypedDict, CATEGORY_KEYWORDS는 dict[str, list[str]]
# 힌트 2: confidence는 float 타입입니다
# 힌트 3:
# class ClassifierState(TypedDict):
#     raw_text: str
#     processed_text: str
#     category: str
#     confidence: float
#     result: str
#
# CATEGORY_KEYWORDS = {
#     "기술": ["AI", "인공지능", ...],
#     ...
# }

# ClassifierState를 정의하세요

# CATEGORY_KEYWORDS를 정의하세요

print("ClassifierState와 CATEGORY_KEYWORDS 정의 완료")

In [ ]:
# 검증 셀
assert 'ClassifierState' in dir(), "ClassifierState가 정의되어야 합니다"
assert 'CATEGORY_KEYWORDS' in dir(), "CATEGORY_KEYWORDS가 정의되어야 합니다"
assert len(CATEGORY_KEYWORDS) >= 3, "최소 3개의 카테고리가 필요합니다"
assert "기술" in CATEGORY_KEYWORDS, "'기술' 카테고리가 있어야 합니다"
assert "경제" in CATEGORY_KEYWORDS, "'경제' 카테고리가 있어야 합니다"
assert "스포츠" in CATEGORY_KEYWORDS, "'스포츠' 카테고리가 있어야 합니다"
print(f"카테고리: {list(CATEGORY_KEYWORDS.keys())}")
print("✅ 실습 1 완료!")

## 실습 2: 3개 노드 함수 구현

다음 3개 노드 함수를 구현하세요:

1. `preprocess(state)`: `raw_text`를 소문자로 변환하고 앞뒤 공백 제거 → `processed_text`
2. `classify(state)`: `processed_text`에서 각 카테고리의 키워드 매칭 수를 세어 가장 많이 매칭된 카테고리와 신뢰도 결정 → `category`, `confidence`
3. `format_result(state)`: `"[{category}] 신뢰도: {confidence:.0%} - '{raw_text}'"` 형식의 문자열 → `result`

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: preprocess는 strip()과 lower()를 사용합니다
# 힌트 2: classify는 각 카테고리의 키워드 중 processed_text에 포함된 것의 수를 셉니다
#         신뢰도 = 매칭 수 / 전체 키워드 수 (단, 0이면 기타로 분류)
# 힌트 3:
# def classify(state: ClassifierState) -> dict:
#     text = state["processed_text"]
#     best_category = "기타"
#     best_score = 0
#     total_keywords = 0
#     for category, keywords in CATEGORY_KEYWORDS.items():
#         if not keywords:
#             continue
#         matches = sum(1 for kw in keywords if kw.lower() in text)
#         if matches > best_score:
#             best_score = matches
#             best_category = category
#             total_keywords = len(keywords)
#     confidence = best_score / total_keywords if total_keywords > 0 and best_score > 0 else 0.0
#     return {"category": best_category, "confidence": confidence}

# preprocess, classify, format_result 함수를 구현하세요

print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
# preprocess 테스트
s = {"raw_text": "  AI와 개발 트렌드  ", "processed_text": "", "category": "", "confidence": 0.0, "result": ""}
r = preprocess(s)
assert r["processed_text"] == "ai와 개발 트렌드", f"전처리 결과가 올바르지 않습니다: {r['processed_text']}"

# classify 테스트
s["processed_text"] = "ai와 개발 트렌드"
r2 = classify(s)
assert r2["category"] == "기술", f"'AI와 개발 트렌드'는 기술 카테고리여야 합니다. 결과: {r2['category']}"
assert r2["confidence"] > 0, "신뢰도가 0보다 커야 합니다"

# format_result 테스트
s.update(r2)
r3 = format_result(s)
assert "기술" in r3["result"], "result에 카테고리가 포함되어야 합니다"
assert "신뢰도" in r3["result"], "result에 신뢰도가 포함되어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: 분류기 그래프 조립 및 테스트

3노드 분류기 그래프를 조립하고 다양한 텍스트로 테스트하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: START → preprocess → classify → format_result → END
# 힌트 2: 초기 상태에서 processed_text, category 등을 빈 문자열로 초기화합니다
# 힌트 3:
# graph = StateGraph(ClassifierState)
# graph.add_node("preprocess", preprocess)
# ... (나머지 노드와 엣지 추가)
# app = graph.compile()

# 그래프를 조립하고 컴파일하세요

# 다음 테스트 텍스트로 실행하세요
test_texts = [
    "AI와 머신러닝 알고리즘 개발 트렌드",
    "주식 시장 금리 인상 투자 전략",
    "축구 월드컵 한국 선수 우승",
    "오늘 날씨가 맑고 좋네요"
]

# 각 텍스트의 분류 결과를 출력하세요

In [ ]:
# 검증 셀
initial = lambda text: {"raw_text": text, "processed_text": "", "category": "", "confidence": 0.0, "result": ""}

r_tech = app.invoke(initial("AI와 머신러닝 알고리즘 개발"))
assert r_tech["category"] == "기술", f"기술 텍스트가 잘못 분류되었습니다: {r_tech['category']}"

r_econ = app.invoke(initial("주식 시장 금리 인상 투자 전략"))
assert r_econ["category"] == "경제", f"경제 텍스트가 잘못 분류되었습니다: {r_econ['category']}"

r_sport = app.invoke(initial("축구 월드컵 선수 우승"))
assert r_sport["category"] == "스포츠", f"스포츠 텍스트가 잘못 분류되었습니다: {r_sport['category']}"

print("✅ 실습 3 완료!")

## 정리

### 오늘 배운 내용
- 3노드 선형 파이프라인: **전처리 → 분류 → 포맷**
- 키워드 딕셔너리(`CATEGORY_KEYWORDS`)를 활용한 규칙 기반 분류
- 신뢰도(confidence) 점수 산출 패턴

### 다음 모듈 안내
**Module 03: 개발 환경** - FakeLLM을 직접 구현하고 테스트합니다.